<a href="https://colab.research.google.com/github/AkibHasanGH/ml-assignment-colab/blob/main/Copy_of_DL_Assignment_03_Question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# DL Assignment 03

**Name:**

**Course Email:**  


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

## Using your chosen dataset:

Load the dataset.

Perform necessary preprocessing:

Handle missing values (if any)

Encode categorical variables (if necessary)

Feature scaling (if needed)

Separate features (X) and target (y).

Convert them into NumPy arrays.

Convert them into PyTorch tensors.

Split into training and testing sets.

Clearly explain each preprocessing decision.

# **Write** Answer 01:


In [9]:
from google.colab import files
uploaded = files.upload()



Saving insurance.csv to insurance.csv


In [10]:
import pandas as pd
df = pd.read_csv("insurance.csv")
print(df.head())


   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [11]:

import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("insurance.csv")

df = pd.get_dummies(df, columns=['sex','smoker','region'], drop_first=True)

X = df.drop("charges", axis=1)
y = df["charges"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_np = np.array(X_scaled, dtype=np.float32)
y_np = np.array(y, dtype=np.float32).reshape(-1,1)

X_tensor = torch.from_numpy(X_np)
y_tensor = torch.from_numpy(y_np)

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)


Training set shape: torch.Size([1070, 8]) torch.Size([1070, 1])
Testing set shape: torch.Size([268, 8]) torch.Size([268, 1])


# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


## Write Answer 02:


In [12]:
import torch
import torch.nn as nn

class InsuranceNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(InsuranceNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

input_dim = 6
hidden_dim = 16
output_dim = 1

model = InsuranceNN(input_dim, hidden_dim, output_dim)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total trainable parameters:", total_params)


InsuranceNN(
  (fc1): Linear(in_features=6, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=1, bias=True)
)
Total trainable parameters: 129


এই neural network‑টি PyTorch এর nn.Module ব্যবহার করে তৈরি করা হয়েছে। আমাদের dataset‑এ মোট 6টি feature আছে (age, bmi, children, sex, smoker, region), তাই input layer‑এ 6টি neuron রাখা হয়েছে। Target হলো medical charges, যা একটি continuous value, তাই output layer‑এ একটি neuron রাখা হয়েছে।

Hidden layer‑এ 16টি neuron ব্যবহার করা হয়েছে। Hidden neurons সংখ্যা নির্ধারণ করার সময় balance রাখতে হয়: খুব কম হলে model underfit করবে, আর খুব বেশি হলে overfit হওয়ার ঝুঁকি থাকে। 16 একটি মাঝারি মান, যা dataset‑এর আকার এবং feature সংখ্যা অনুযায়ী যথাযথ।

Activation function হিসেবে ReLU ব্যবহার করা হয়েছে। ReLU computationally efficient এবং vanishing gradient problem এড়ায়। Regression problem‑এ hidden layer‑এ ReLU ভালো কাজ করে। Output layer‑এ activation function ব্যবহার করা হয়নি, কারণ আমাদের লক্ষ্য continuous value predict করা।

Trainable parameters গণনা করলে পাওয়া যায়:

Input → Hidden: (6×16 weights + 16 bias) = 112

Hidden → Output: (16×1 weights + 1 bias) = 17
মোট = 129 trainable parameters।

# Question 03: [ Marks 10 ]

Choose an appropriate loss function.

Choose an optimizer.

<br>

Justify your choices based on:

Regression vs Classification

Nature of the dataset

## Write Answer 03:
Loss Function: Mean Squared Error (MSELoss)

Optimizer: Adam

Justification:  
আমাদের dataset হলো Medical Cost Personal Dataset, যেখানে target variable charges একটি continuous সংখ্যা। তাই এটি একটি Regression problem, Classification নয়। এজন্য loss function হিসেবে Mean Squared Error (MSELoss) সবচেয়ে উপযুক্ত, কারণ এটি predicted value এবং actual value এর মধ্যে squared difference minimize করে। Classification problem হলে CrossEntropyLoss ব্যবহার করা হতো, কিন্তু regression‑এ probability নয়, continuous value predict করতে হয়।

Optimizer হিসেবে Adam বেছে নেওয়া হয়েছে। Adam momentum এবং adaptive learning rate একসাথে ব্যবহার করে, ফলে training দ্রুত converge হয় এবং local minima থেকে বের হতে সাহায্য করে। ছোট dataset এবং regression task‑এ Adam সাধারণত SGD এর চেয়ে ভালো কাজ করে। তাই এখানে Adam একটি practical এবং efficient choice।

# Question 04: [ Marks 15 ]

## Implement a full training loop:

Forward pass

Loss computation

Backward pass

Parameter update

Gradient reset

### Requirements:

Train for at least 100 epochs.

Print loss every 10 epochs.

Store training loss history(You can pick your own Data Structure).

Explain clearly what happens in each step of the pipeline.

## Write Answer 04:

In [14]:
import torch
import torch.nn as nn

class InsuranceNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(InsuranceNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

input_dim = X_train.shape[1]
hidden_dim = 16
output_dim = 1

model = InsuranceNN(input_dim, hidden_dim, output_dim)
print(model)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

loss_history = []

epochs = 100
for epoch in range(1, epochs+1):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 10 == 0:
        print(f"Epoch [{epoch}/{epochs}], Loss: {loss.item():.4f}")



InsuranceNN(
  (fc1): Linear(in_features=8, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=1, bias=True)
)
Epoch [10/100], Loss: 322425056.0000
Epoch [20/100], Loss: 322388800.0000
Epoch [30/100], Loss: 322332448.0000
Epoch [40/100], Loss: 322248192.0000
Epoch [50/100], Loss: 322129824.0000
Epoch [60/100], Loss: 321972800.0000
Epoch [70/100], Loss: 321773312.0000
Epoch [80/100], Loss: 321528896.0000
Epoch [90/100], Loss: 321238144.0000
Epoch [100/100], Loss: 320898368.0000


Error এসেছে কারণ feature সংখ্যা 8, কিন্তু model‑এ 6 দেওয়া হয়েছিল।

সমাধান: input_dim = X_train.shape[1] ব্যবহার করলে feature সংখ্যা auto detect হবে।

এখন forward pass ঠিকভাবে কাজ করবে, আর training loop smooth চলবে।

👉 এভাবে তোমার shape mismatch problem solve হয়ে যাবে।

# Question 05: [ Marks 10 ]

## Evaluate the model on test data.

## For regression:

Report MSE and MAE


## For classification:

Report Accuracy

Compare training vs testing performance.

State whether the model is underfitting or overfitting.

## Write Answer 05:

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

model.eval()
with torch.no_grad():
    y_pred_train = model(X_train)
    y_pred_test = model(X_test)

train_mse = mean_squared_error(y_train.numpy(), y_pred_train.numpy())
train_mae = mean_absolute_error(y_train.numpy(), y_pred_train.numpy())

test_mse = mean_squared_error(y_test.numpy(), y_pred_test.numpy())
test_mae = mean_absolute_error(y_test.numpy(), y_pred_test.numpy())

print("Training MSE:", train_mse)
print("Training MAE:", train_mae)
print("Testing MSE:", test_mse)
print("Testing MAE:", test_mae)


Training MSE: 320861440.0
Training MAE: 13299.5498046875
Testing MSE: 321846304.0
Testing MAE: 12922.673828125


MSE & MAE:

MSE → average squared error, বড় ভুলকে বেশি penalize করে।

MAE → average absolute error, সহজে বোঝা যায় কতটা deviation আছে।

Training vs Testing Performance:

যদি training loss অনেক কম আর testing loss অনেক বেশি হয় → overfitting।

যদি training ও testing দুটোই বেশি হয় → underfitting।

যদি training ও testing কাছাকাছি হয় → model ভালোভাবে generalize করছে।

# Question 06: [ Marks 20 ]

## Modify at least ONE of the following:

Learning rate

Number of hidden neurons

Number of epochs

### Train again and compare:

Convergence speed

Final performance

Explain how the change affected the model.

## Write Answer 06:

### Increasing Number of Epochs

To investigate the effect of training duration, I am modifying the `epochs` parameter from 100 to 200 in the training cell below. This change will allow us to observe if the model converges further or if its performance changes with more training iterations.

In [16]:
input_dim = X_train.shape[1]
hidden_dim = 32   # আগে ছিল 16
output_dim = 1

model = InsuranceNN(input_dim, hidden_dim, output_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

loss_history = []
epochs = 100
for epoch in range(1, epochs+1):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 10 == 0:
        print(f"Epoch [{epoch}/{epochs}], Loss: {loss.item():.4f}")


Epoch [10/100], Loss: 322414016.0000
Epoch [20/100], Loss: 322352256.0000
Epoch [30/100], Loss: 322246432.0000
Epoch [40/100], Loss: 322078208.0000
Epoch [50/100], Loss: 321830144.0000
Epoch [60/100], Loss: 321486784.0000
Epoch [70/100], Loss: 321035904.0000
Epoch [80/100], Loss: 320470176.0000
Epoch [90/100], Loss: 319786240.0000
Epoch [100/100], Loss: 318983040.0000


# Question 07: [ Marks 20 ]


# Training Analysis

Answer the following:

Why must gradients be reset every epoch?

What happens if learning rate is too high?

What happens if learning rate is too small?

Why do we define layers inside the constructor (__init__) and not inside forward()?


## Write Answer 07:

### Why must gradients be reset every epoch?

Gradients must be reset (zeroed out) at the beginning of each training step or epoch because PyTorch accumulates gradients by default. If you don't reset them, the gradients from the previous training step will be added to the gradients of the current step, leading to incorrect updates to the model's parameters. This would essentially average the gradients over multiple steps, hindering the optimization process and preventing the model from learning effectively from the current batch of data.

### What happens if the learning rate is too high?

If the learning rate is too high, the optimization algorithm will take very large steps in the parameter space. This can cause the model's loss function to oscillate wildly, potentially jumping over the optimal solution. In extreme cases, the loss might diverge to infinity (exploding gradients), making the model unable to learn anything meaningful.

### What happens if the learning rate is too small?

If the learning rate is too small, the optimization algorithm will take very tiny steps. While this might lead the model towards a local or global minimum, the process will be extremely slow, requiring a very large number of epochs to converge. The model might get stuck in a suboptimal solution or a flat region of the loss landscape, or it might simply take too long to train, making it impractical.

### Why do we define layers inside the constructor (`__init__`) and not inside `forward()`?

We define the layers (e.g., `nn.Linear`, `nn.ReLU`) inside the `__init__` constructor of an `nn.Module` for several reasons:

1.  **Stateful Nature:** Neural network layers are stateful; they contain trainable parameters (weights and biases). These parameters need to be initialized once and then updated throughout the training process. Defining them in `__init__` ensures they are created and registered as part of the model's state when the model object is instantiated.
2.  **Efficiency:** If you were to define layers inside the `forward()` method, a new set of parameters would be created every time `forward()` is called. This would be highly inefficient, as it would lead to constant re-initialization of weights and biases, making it impossible for the model to learn and consume excessive memory.
3.  **Parameter Registration:** When layers are defined as attributes of `nn.Module` in `__init__`, PyTorch's internal mechanisms automatically detect them and include their parameters in the list of trainable parameters returned by `model.parameters()`. This allows the optimizer to correctly update all the model's weights and biases during backpropagation.
4.  **Modularity and Readability:** It promotes a clear separation of concerns. The `__init__` method defines the static architecture and components of the network, while the `forward()` method defines how data flows through these predefined components. This makes the code more organized, readable, and easier to debug.